# 00 — Setup check (Translation Tangles reproduction)

Run once on each Kaggle account with **Accelerator: GPU T4 x2** and **Internet: On**.
Attach the secrets `HF_TOKEN`, `GEMINI_API_KEY`, `GROQ_API_KEY` (Add-ons → Secrets).

It checks the GPU, the secrets, access to the gated models, every dataset the paper uses,
and which Groq / Gemini models your keys can reach. Nothing here prints a key.
At the end it writes `/kaggle/working/setup_report.json` — download that file and send it back.

In [ ]:
import json, os, platform, traceback
report = {"python": platform.python_version()}

def ok(section, key, value):
    report.setdefault(section, {})[key] = value
    print(f"[{section}] {key}: {value}")

## 1. GPU

In [ ]:
try:
    import torch
    n = torch.cuda.device_count()
    ok("gpu", "count", n)
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        ok("gpu", f"device_{i}", f"{p.name}, {p.total_memory / 2**30:.1f} GB")
except Exception as e:
    ok("gpu", "error", repr(e))

## 2. Secrets (only reports set / missing)

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = {}
for name in ["HF_TOKEN", "GEMINI_API_KEY", "GROQ_API_KEY"]:
    try:
        secrets[name] = UserSecretsClient().get_secret(name)
        ok("secrets", name, "set")
    except Exception:
        secrets[name] = None
        ok("secrets", name, "MISSING")

## 3. Gated model access on HuggingFace

In [ ]:
from huggingface_hub import auth_check, login
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

if secrets["HF_TOKEN"]:
    login(token=secrets["HF_TOKEN"], add_to_git_credential=False)

MODELS = [
    # general benchmark, run on Kaggle
    "google/gemma-7b-it",
    "google/gemma-2-9b-it",
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "allenai/OLMo-1B-hf",
    "microsoft/Phi-3.5-mini-instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    # domain benchmark, 32B class
    "Qwen/Qwen2.5-32B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
]
for repo in MODELS:
    try:
        auth_check(repo)
        ok("hf_models", repo, "accessible")
    except GatedRepoError:
        ok("hf_models", repo, "GATED - accept the licence on its HF page")
    except RepositoryNotFoundError:
        ok("hf_models", repo, "NOT FOUND")
    except Exception as e:
        ok("hf_models", repo, f"error: {e!r}"[:200])

## 4. Datasets

Streams two rows from each source (no full download) and records the configs, columns,
split sizes and one example, so the data-prep notebook can be written against the real schema.

In [ ]:
from datasets import load_dataset, get_dataset_config_names, load_dataset_builder
import datasets
ok("datasets", "library_version", datasets.__version__)

def probe(repo, config=None, split=None):
    tag = f"{repo} | {config} | {split}"
    entry = {}
    try:
        try:
            info = load_dataset_builder(repo, config).info
            entry["splits"] = {k: v.num_examples for k, v in (info.splits or {}).items()}
        except Exception as e:
            entry["splits"] = f"unknown ({type(e).__name__})"
        ds = load_dataset(repo, config, split=split, streaming=True)
        rows = list(ds.take(2))
        entry["columns"] = list(rows[0].keys()) if rows else []
        entry["example"] = str(rows[0])[:400] if rows else None
    except Exception as e:
        entry["error"] = f"{type(e).__name__}: {e}"[:300]
    ok("datasets", tag, entry)

def configs(repo):
    try:
        names = get_dataset_config_names(repo)
    except Exception as e:
        names = f"error: {type(e).__name__}: {e}"[:300]
    ok("dataset_configs", repo, names)
    return names if isinstance(names, list) else []

# General benchmark
for cfg in configs("wmt/wmt19"):
    probe("wmt/wmt19", cfg, "validation")
for cfg in ["et-en", "tr-en"]:
    probe("wmt/wmt18", cfg, "test")
configs("wmt/wmt18")
configs("csebuetnlp/BanglaNMT")
probe("csebuetnlp/BanglaNMT", None, "test")

# Domain benchmark
lit = configs("Nothingger/kaz-rus-eng-literature-parallel-corpus")
probe("Nothingger/kaz-rus-eng-literature-parallel-corpus", lit[0] if lit else None, "train")
med = configs("qanastek/ELRC-Medical-V2")
for cfg in [c for c in med if any(l in c for l in ["fr", "hr", "it", "pl", "de", "es", "pt", "fi"])][:8] or med[:1]:
    probe("qanastek/ELRC-Medical-V2", cfg, "train")
law = configs("coastalcph/multi_eurlex")
probe("coastalcph/multi_eurlex", "all_languages" if "all_languages" in law else (law[0] if law else None), "test")

## 5. Groq: which models can the key reach?

In [ ]:
import requests
if secrets["GROQ_API_KEY"]:
    try:
        r = requests.get("https://api.groq.com/openai/v1/models",
                         headers={"Authorization": f"Bearer {secrets['GROQ_API_KEY']}"}, timeout=30)
        ok("groq", "status", r.status_code)
        if r.ok:
            ok("groq", "models", sorted(m["id"] for m in r.json().get("data", [])))
    except Exception as e:
        ok("groq", "error", repr(e))
else:
    ok("groq", "status", "no key")

## 6. Gemini: embedding + judge model available, one tiny embedding call

In [ ]:
if secrets["GEMINI_API_KEY"]:
    H = {"x-goog-api-key": secrets["GEMINI_API_KEY"]}
    base = "https://generativelanguage.googleapis.com/v1beta"
    try:
        r = requests.get(f"{base}/models?pageSize=200", headers=H, timeout=30)
        ok("gemini", "list_status", r.status_code)
        if r.ok:
            names = [m["name"] for m in r.json().get("models", [])]
            ok("gemini", "embedding_models", [n for n in names if "embedding" in n])
            ok("gemini", "flash_models", [n for n in names if "2.5-flash" in n])
        r = requests.post(f"{base}/models/gemini-embedding-001:embedContent", headers=H, timeout=30,
                          json={"content": {"parts": [{"text": "The church will be visible from afar."}]}})
        ok("gemini", "embed_status", r.status_code)
        if r.ok:
            ok("gemini", "embed_dim", len(r.json()["embedding"]["values"]))
        else:
            ok("gemini", "embed_error", r.text[:300])
    except Exception as e:
        ok("gemini", "error", repr(e))
else:
    ok("gemini", "status", "no key")

## 7. Save the report

In [ ]:
with open("/kaggle/working/setup_report.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print("Saved /kaggle/working/setup_report.json — download it from the Output panel and send it back.")